# RDLNet 两阶段训练（Colab 最小可跑）

1. **挂载 Google Drive**：**数据与 checkpoint** 放在 `DRIVE_ROOT`（持久）。**代码**每次在第 2 格重新 clone 到 **`/content/doc-scan`**（本机，断连即无，不占 Drive）。  
2. **拉代码 / 装依赖**：第 2 格 **带 submodule** clone；私有库改 `REPO_URL`。  
3. **阶段 1 蒸馏** → 权重存 `DRIVE_ROOT/checkpoints`。  
4. **阶段 2 RDLNet**：`--resume` 续跑；`--save-every-steps` 防中途断线。

把 `DRIVE_ROOT` 改成你在 Drive 里放数据与权重的目录。

In [1]:
# @title 1. 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# @title 2. 每次重新 clone 到 /content + 依赖（只改 DRIVE_ROOT、REPO_URL）
import os, sys, subprocess, shutil

_WORK = '/content'  # 删 REPO_DIR 前必须回到此目录，否则 cwd 在已删路径里 git/! 会报错
DRIVE_ROOT = '/content/drive/MyDrive'  # 数据与 checkpoint；改成你的子目录
REPO_DIR = '/content/doc-scan'  # 代码放本机，每次先删再 clone
REPO_URL = 'https://github.com/weinixuehao/rdlnet.git'  # 私有库见上方说明

os.chdir(_WORK)
os.makedirs(DRIVE_ROOT, exist_ok=True)
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.check_call(
    ['git', 'clone', '--recurse-submodules', REPO_URL, REPO_DIR],
    cwd=_WORK,
)
subprocess.check_call(
    ['git', '-C', REPO_DIR, 'submodule', 'update', '--init', '--recursive'],
    cwd=_WORK,
)

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
req = os.path.join(REPO_DIR, 'requirements.txt')
if os.path.isfile(req):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', req])
seg = os.path.join(REPO_DIR, 'segment-anything')
if os.path.isdir(seg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', seg])
print('OK cwd=', os.getcwd(), '| DRIVE_ROOT=', DRIVE_ROOT)


In [ ]:
# @title 3. 阶段 1：Light-SAM 蒸馏（断线后用 --resume 同一路径）
import subprocess, sys, os

IMAGE_DIR = os.path.join(DRIVE_ROOT, 'images_unlabeled')   # 无标注图片文件夹
TEACHER = os.path.join(DRIVE_ROOT, 'sam_vit_h_4b8939.pth')
OUT = os.path.join(DRIVE_ROOT, 'checkpoints', 'distill_stage1.pt')
os.makedirs(os.path.dirname(OUT), exist_ok=True)

cmd = [
    sys.executable, 'train_distill.py',
    '--image-dir', IMAGE_DIR,
    '--teacher-checkpoint', TEACHER,
    '--output', OUT,
    '--epochs', '1',
    '--batch-size', '1',
    '--img-size', '1024',
    '--save-every-steps', '200',
    # 续跑时取消下面注释并指向已有 pt
    # '--resume', OUT,
    # '--epochs', '10',
]
subprocess.check_call(cmd, cwd=REPO_DIR)

In [ ]:
# @title 4. 阶段 2：RDLNet（需 annotations.json；骨干用阶段 1 权重）
import subprocess, sys, os

ANN = os.path.join(DRIVE_ROOT, 'annotations.json')
IMG_ROOT = DRIVE_ROOT  # JSON 里 file_name / mask 相对此根目录
DISTILL_CKPT = os.path.join(DRIVE_ROOT, 'checkpoints', 'distill_stage1.pt')
OUT2 = os.path.join(DRIVE_ROOT, 'checkpoints', 'rdlnet.pt')

cmd = [
    sys.executable, 'train_rdlnet.py',
    '--annotations', ANN,
    '--image-root', IMG_ROOT,
    '--distill-checkpoint', DISTILL_CKPT,
    '--output', OUT2,
    '--epochs', '5',
    '--batch-size', '1',
    '--img-size', '1024',
    '--save-every-steps', '100',
    # 续跑：
    # '--resume', OUT2,
    # '--epochs', '20',
]
subprocess.check_call(cmd, cwd=REPO_DIR)

## 断线后续跑

- **阶段 1**：把 `train_distill.py` 加上 `--resume` 指向 `distill_stage1.pt`（或 `distill_stage1_latest.pt`），`--epochs` 写「还要训几轮」。  
- **阶段 2**：`--resume` 指向 `rdlnet.pt` 或 `rdlnet_latest.pt`，**不要**再带 `--distill-checkpoint`。

显存不够时先把 `--img-size` 改为 `512`，阶段 1 与 2 必须一致。